# Supplementary Figure — four-condition paragraph-scrambled sensitivity

This notebook reproduces the four-condition sensitivity analysis using the
exported tables from the paragraph-scrambled validation.

Because the paragraph condition has a different time series length and these
models are fit within condition, this figure is a sensitivity analysis rather
than a replacement for the primary shared-basis results.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

DATA_DIR = Path("four_condition_ratio_decoding_outputs")

RAW_CSV = DATA_DIR / "raw_htfa_summary.csv"
PCA_CSV = DATA_DIR / "pca_summary_with_retention.csv"
AA_CSV = DATA_DIR / "aa_full_summary_with_retention.csv"

CONDITIONS = ["intact", "paragraph", "word", "rest"]
COLORS = {
    "intact": "#7B3294",
    "paragraph": "#E69F00",
    "word": "#2CA25F",
    "rest": "#222222",
}
LABELS = {
    "intact": "Intact",
    "paragraph": "Paragraph-scrambled",
    "word": "Word-scrambled",
    "rest": "Rest",
}

OUTPUT_DIR = Path("outputs/paper_supplement")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Supplement_four_condition_paragraph_validation.png"
FIG_PDF = OUTPUT_DIR / "Supplement_four_condition_paragraph_validation.pdf"

for p in [RAW_CSV, PCA_CSV, AA_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {p}. Run the four-condition ratio-matched analysis first."
        )

raw4 = pd.read_csv(RAW_CSV)
pca4 = pd.read_csv(PCA_CSV)
aa4 = pd.read_csv(AA_CSV)

In [ ]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=(11.3, 8.2),
    sharex=True,
)

for ax, condition, letter in zip(
    axes.ravel(),
    CONDITIONS,
    list("ABCD"),
):
    color = COLORS[condition]

    r = raw4[
        raw4["condition"].astype(str) == condition
    ].iloc[0]

    ax.axhline(
        r["mean"],
        color=color,
        linestyle=":",
        linewidth=1.8,
    )

    ax.axhline(
        r["chance"],
        color="0.70",
        linestyle=":",
        linewidth=1.1,
    )

    p = (
        pca4[
            pca4["condition"].astype(str) == condition
        ]
        .sort_values("actual_ratio")
    )

    ax.errorbar(
        p["actual_ratio"],
        p["mean"],
        yerr=p["err"],
        color=color,
        linestyle="-.",
        marker="^",
        markersize=3.6,
        capsize=2,
        linewidth=1.8,
    )

    for analysis_type, ls, marker in [
        ("spatial", "-", "s"),
        ("temporal", "--", "o"),
    ]:
        a = (
            aa4[
                (aa4["condition"].astype(str) == condition)
                & (aa4["analysis_type"].astype(str) == analysis_type)
            ]
            .sort_values("actual_ratio")
        )

        ax.errorbar(
            a["actual_ratio"],
            a["mean"],
            yerr=a["err"],
            color=color,
            linestyle=ls,
            marker=marker,
            markersize=3.6,
            capsize=2,
            linewidth=1.8,
        )

    ax.set_title(LABELS[condition], fontweight="bold")
    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel("Timepoint-identification accuracy")
    ax.tick_params(labelsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.text(
        -0.10, 1.04, letter,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
    )

legend_handles = [
    Line2D([0], [0], color="0.25", linestyle="-.", marker="^", label="PCA"),
    Line2D([0], [0], color="0.25", linestyle="-", marker="s", label="Spatial MS-AA"),
    Line2D([0], [0], color="0.25", linestyle="--", marker="o", label="Temporal MS-AA"),
    Line2D([0], [0], color="0.25", linestyle=":", label="Raw HTFA"),
    Line2D([0], [0], color="0.70", linestyle=":", label="Chance"),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=5,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)

fig.suptitle(
    "Four-condition sensitivity analysis",
    fontsize=16,
    fontweight="bold",
)

fig.subplots_adjust(bottom=0.10, top=0.93, hspace=0.30, wspace=0.24)

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)

In [ ]:
# Optional second panel set: fraction of raw above-chance information retained

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11.3, 8.2),
    sharex=True,
)

for ax, condition in zip(axes.ravel(), CONDITIONS):
    color = COLORS[condition]

    ax.axhline(
        1.0,
        color="0.50",
        linestyle=":",
        linewidth=1.2,
    )

    p = (
        pca4[
            pca4["condition"].astype(str) == condition
        ]
        .sort_values("actual_ratio")
    )

    ax.plot(
        p["actual_ratio"],
        p["fraction_raw_above_chance_retained"],
        color=color,
        linestyle="-.",
        marker="^",
        markersize=3.6,
        linewidth=1.8,
    )

    for analysis_type, ls, marker in [
        ("spatial", "-", "s"),
        ("temporal", "--", "o"),
    ]:
        a = (
            aa4[
                (aa4["condition"].astype(str) == condition)
                & (aa4["analysis_type"].astype(str) == analysis_type)
            ]
            .sort_values("actual_ratio")
        )

        ax.plot(
            a["actual_ratio"],
            a["fraction_raw_above_chance_retained"],
            color=color,
            linestyle=ls,
            marker=marker,
            markersize=3.6,
            linewidth=1.8,
        )

    ax.set_title(LABELS[condition], fontweight="bold")
    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel("Fraction of raw above-chance information retained")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

ret_png = OUTPUT_DIR / "Supplement_four_condition_paragraph_retention.png"
ret_pdf = OUTPUT_DIR / "Supplement_four_condition_paragraph_retention.pdf"

fig.suptitle(
    "Four-condition fractional information retention",
    fontsize=16,
    fontweight="bold",
)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(ret_png, dpi=300, bbox_inches="tight")
fig.savefig(ret_pdf, bbox_inches="tight")
plt.show()

print("Saved:", ret_png)
print("Saved:", ret_pdf)